In [ ]:
!pip install transformers
!pip install 'accelerate>=1.1.0'
!pip install --upgrade httpx httpcore

In [ ]:
import os
print("현재 위치:", os.getcwd())
print("해당 경로에 파일이 있는가?:", os.path.exists('../augmentation_data/train_augmented_ALL.csv'))
print("현재 폴더 내 파일들:", os.listdir('.'))

In [9]:
import pandas as pd
import torch
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# ---------------------------------------------
# 0. 메모리 초기화
# ---------------------------------------------
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------
train_combined_path = './augmentation_data/train_augmented_ALL.csv'
df = pd.read_csv(train_combined_path)
df = df.dropna(subset=['conversation', 'label'])

# Train / Validation 분리 (8:2)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# ---------------------------------------------
# 2. 모델 및 토크나이저 로드 (하이퍼파라미터 원본 유지)
# ---------------------------------------------
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 5          
LEARNING_RATE = 1e-5

MODEL_NAME = "beomi/KcELECTRA-base-v2022" 
num_labels = len(df['label'].unique()) 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# ---------------------------------------------
# 3. Custom Dataset 클래스
# ---------------------------------------------
class KcElectraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = KcElectraDataset(train_df['conversation'].values, train_df['label'].values, tokenizer, MAX_LENGTH)
val_dataset = KcElectraDataset(val_df['conversation'].values, val_df['label'].values, tokenizer, MAX_LENGTH)

# ---------------------------------------------
# 4. 평가지표 계산 함수
# ---------------------------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    
    return {
        'accuracy': acc,
        'f1_macro': f1
    }

# ---------------------------------------------
# 5. Trainer 셋팅 및 학습 (Training)
# ---------------------------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,                    # EPOCHS = 5
    per_device_train_batch_size=32,        # BATCH_SIZE = 32
    per_device_eval_batch_size=32,
    learning_rate=2e-5,                    # LEARNING_RATE = 1e-5
    weight_decay=0.01,                     # Weight Decay = 0.01
    adam_epsilon=1e-5,                     # Epsilon = 1e-5
    max_grad_norm=1.0,                     # Max Grad Norm = 1.0
    warmup_ratio=0.1,                      # SCHEDULER = 10% Warmup
    lr_scheduler_type="linear",            # SCHEDULER = Linear
    seed=42,                               # SEED = 42
    eval_strategy="epoch",                 # (에러 수정됨) 에포크마다 평가
    save_strategy="epoch",                 
    logging_steps=50,                      
    report_to="none"                       
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("▶ [1/2] 모델 학습을 시작합니다...\n")
trainer.train()

# ==========================================
# 6. 최종 평가 및 리포트 출력 (Evaluation)
# ==========================================
print("\n▶ [2/2] 최종 모델 성능 평가를 진행합니다...")

# 기본 metrics 확인 (eval_loss, eval_accuracy 등)
eval_results = trainer.evaluate()

print("-" * 50)
print("🎯 [최종 검증 세트(Validation Set) 평가 결과]")
print(f" - Loss (손실): {eval_results['eval_loss']:.4f}")
print(f" - Accuracy (정확도): {eval_results['eval_accuracy'] * 100:.2f}%")
print(f" - F1 Score (Macro): {eval_results['eval_f1_macro']:.4f}")
print("-" * 50)

print("\n▶ 검증 데이터셋 상세 예측 중...")
# val_dataset에 대해 예측을 수행하여 로짓(logits) 추출
output = trainer.predict(val_dataset)
preds = np.argmax(output.predictions, axis=-1)

# 타겟 이름 맵핑 (0~4번 인덱스 순서에 맞게 설정)
target_names = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']

print("\n" + "="*60)
print("🎯 [KcELECTRA 상세 성능 리포트]")
print("="*60)
# 실제 정답(val_df['label'])과 모델의 예측값(preds) 비교
print(classification_report(val_df['label'].values, preds, target_names=target_names, zero_division=0))
print("="*60)

# 학습된 모델 및 토크나이저 안전하게 저장
save_directory = "./best_kcelectra_model2"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"\n✅ 학습 완료! 모델과 토크나이저가 '{save_directory}' 폴더에 안전하게 저장되었습니다.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

▶ [1/2] 모델 학습을 시작합니다...



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.456476,1.060101,0.753764,0.761415
2,0.735665,0.529296,0.885740,0.890310
3,0.392933,0.347673,0.916740,0.921268
4,0.268848,0.280114,0.927369,0.931392
5,0.231047,0.262007,0.930912,0.934476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.231047,0.262007,5,0.930912,0.934476


--------------------------------------------------
🎯 [최종 검증 세트(Validation Set) 평가 결과]
 - Loss (손실): 0.2620
 - Accuracy (정확도): 93.09%
 - F1 Score (Macro): 0.9345
--------------------------------------------------

▶ 검증 데이터셋 상세 예측 중...



🎯 [KcELECTRA 상세 성능 리포트]
              precision    recall  f1-score   support

       협박 대화       0.89      0.93      0.91       218
       갈취 대화       0.92      0.92      0.92       248
 직장 내 괴롭힘 대화       0.95      0.95      0.95       225
   기타 괴롭힘 대화       0.92      0.88      0.90       259
       일반 대화       0.99      1.00      0.99       179

    accuracy                           0.93      1129
   macro avg       0.93      0.94      0.93      1129
weighted avg       0.93      0.93      0.93      1129



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ 학습 완료! 모델과 토크나이저가 './best_kcelectra_model2' 폴더에 안전하게 저장되었습니다.


### submission

In [10]:
import pandas as pd
from transformers import pipeline

# 1. 파일 로드 및 모델 준비
test_df = pd.read_csv("data/test.csv")
submission_df = pd.read_csv("data/submission_default.csv")

classifier = pipeline(
    "text-classification", 
    model=trainer.model, 
    tokenizer=tokenizer, 
    device=0 
)

# 2. 추론 실행 
print(f"총 {len(test_df)}건 추론 중...")
results = classifier(test_df['conversation'].tolist(), batch_size=16)

# 3. 결과 정리 (핵심 변경 부분!)
# res['label']은 'LABEL_0' 형태로 나오므로, 문자열에서 숫자만 추출(int)합니다.
# 예: 'LABEL_0' -> 0
predictions = [int(res['label'].split('_')[-1]) for res in results]

# 4. submission 양식에 채우기
submission_df['class'] = predictions

# 5. 저장 (숫자만 들어가므로 인코딩 옵션은 빼도 무방합니다)
output_path = "data/submission_KcELECTRA3.csv"
submission_df.to_csv(output_path, index=False)

print("-" * 50)
print(f"✅ 제출 파일 생성 완료: {output_path}")

# 6. 결과 확인
print(submission_df.head(10))

총 500건 추론 중...
--------------------------------------------------
✅ 제출 파일 생성 완료: data/submission_KcELECTRA3.csv
     idx  class
0  t_000      1
1  t_001      2
2  t_002      2
3  t_003      4
4  t_004      3
5  t_005      0
6  t_006      0
7  t_007      1
8  t_008      4
9  t_009      1
